# Generate a TRAILS Data Package with premise (Tutorial)

**Authors**: Romain Sacchi (PSI)

**Contact**: romain.sacchi@psi.ch

**premise version note:** `premise.TrailsDataPackage` is available starting with `premise` v2.5.0. If that version is not available in your environment yet, install the `trails` branch from GitHub:

```bash
pip install git+https://github.com/polca/premise.git@trails
```

## Purpose

This notebook shows how to build a premise pathways datapackage and export it as a
TRAILS-compatible datapackage.

You will learn how to:

1. Configure a Brightway project and a premise scenario.
2. Choose explicit export years or rely on the default IAM year grid.
3. Create and export a datapackage.
4. Pass the resulting zip file to Notebook 2.2.

## Prerequisites

- `premise`, `bw2data`, and `bw2io` installed in the active environment.
- A Brightway project with an imported ecoinvent source database.
- Valid ecoinvent credentials if the source database still needs to be imported.
- A valid `IAM_FILES_KEY` for the default premise scenario files.

## Key clarification about years

With `TrailsDataPackage`, you can pass `years=[...]` explicitly.

If you do not pass `years`, premise uses the default IAM year grid, which
typically runs from 2005 to 2100 in 5-year steps.

In [ ]:
import os
from pathlib import Path

import bw2data
import bw2io
from premise import TrailsDataPackage, clear_inventory_cache

## 1. Configure the export

Keep secrets out of the notebook itself whenever possible. The example below
reads credentials and the IAM key from environment variables.

You can either pass a custom `YEARS` list or omit it and let premise use the
default IAM year grid.

In [ ]:
PROJECT_NAME = "your_bw_project"
EI_VERSION = "3.11"
SYSTEM_MODEL = "cutoff"
SOURCE_DB = f"ecoinvent-{EI_VERSION}-{SYSTEM_MODEL}"
BIOSPHERE_NAME = "biosphere"

MODEL = "remind-eu"
PATHWAY = "SSP2-PkBudg1000"
YEARS = [2020, 2030, 2040, 2050, 2100]

# To use the default IAM years instead, set YEARS = None.
# premise then typically uses 2005..2100 in 5-year steps.

EXPORT_NAME = "trails_example_remind_eu"
IAM_FILES_KEY = os.environ.get("IAM_FILES_KEY", "replace-with-your-premise-key")
EI_USERNAME = os.environ.get("EI_USERNAME")
EI_PASSWORD = os.environ.get("EI_PASSWORD")

## 2. Select the Brightway project

If the source ecoinvent database is not already present in the project, import
it first. Otherwise, this cell simply switches to the project and prints the
known databases.

In [ ]:
bw2data.projects.set_current(PROJECT_NAME)

if SOURCE_DB not in bw2data.databases:
    if not EI_USERNAME or not EI_PASSWORD:
        raise ValueError(
            "Set EI_USERNAME and EI_PASSWORD before importing ecoinvent."
        )

    bw2io.import_ecoinvent_release(
        version=EI_VERSION,
        system_model=SYSTEM_MODEL,
        username=EI_USERNAME,
        password=EI_PASSWORD,
        biosphere_name=BIOSPHERE_NAME,
    )

sorted(bw2data.databases)

## 3. Define the scenario and optional years

`TrailsDataPackage` expects scenario definitions without a year. The years are
passed separately via the optional `years` argument.

In [ ]:
clear_inventory_cache()

scenarios = [
    {"model": MODEL, "pathway": PATHWAY}
]

print("Scenarios:", scenarios)
print(
    "Years:",
    YEARS if YEARS is not None else "premise default IAM years (typically 2005..2100 every 5 years)",
)

## 4. Create the datapackage object

The `key` below is the IAM decryption key used by premise for the bundled
scenario files. Replace the fallback string or, better, set `IAM_FILES_KEY` in
your shell before launching Jupyter.

If `YEARS` is `None`, the `years` argument is omitted and premise falls back to
its default IAM year grid.

In [ ]:
dp_kwargs = dict(
    scenarios=scenarios,
    source_db=SOURCE_DB,
    source_version=EI_VERSION,
    system_model=SYSTEM_MODEL,
    biosphere_name=BIOSPHERE_NAME,
    key=IAM_FILES_KEY,
    use_absolute_efficiency=True,
)

if YEARS is not None:
    dp_kwargs["years"] = YEARS

dp = TrailsDataPackage(**dp_kwargs)

## 5. Create the datapackage

`create_datapackage()` applies the transformations and writes a zipped
datapackage. If you want only selected sectors, pass `transformations=[...]`.

In [ ]:
dp.create_datapackage(
    name=EXPORT_NAME,
    # transformations=["electricity", "cars"],
)

## 6. Locate the exported datapackage

`TrailsDataPackage` writes the zip file in the current working directory.
Use the resulting path in Notebook 2.2.

In [ ]:
datapackage_path = Path.cwd() / f"{EXPORT_NAME}.zip"
print(datapackage_path)
print("Exists:", datapackage_path.exists())

## Common pitfalls

- If you need more temporal detail in the underlying export, pass more years in
  `YEARS`.
- If you leave `YEARS` unset, premise typically uses all IAM years from 2005 to
  2100 in 5-year steps.
- Move on to Notebook 2.2 once the zip file exists.